# 1.4 — Design Phases, Gates, and the Constraint Diagram

Aircraft development is a structured process that moves from vague operational need to certified product through a sequence of increasingly detailed phases.
The industry has refined this sequence over decades, and while companies use different labels, the underlying logic is shared across Boeing, Airbus, Lockheed, and every major aerospace prime {cite}`raymer2018` {cite}`torenbeek2013`.

Understanding the phases matters for more than organisational tidiness.
Each phase has a different *purpose*, demands different *tools*, and carries a different *cost of change* — the penalty for revising a decision made earlier.
Getting the architecture wrong in conceptual design costs days; getting it wrong after production tooling has been cut costs hundreds of millions of dollars {cite}`raymer2018`.

## The five-phase model

Modern practice divides development into five phases (numbered 0–4 in some frameworks, A–E in others):

```{dropdown} Phase 0 — Mission analysis and requirements definition
**Question:** *What problem are we actually solving?*

Before any aircraft shape exists, the customer and the engineering team must agree on what the aircraft needs to *do*.
This phase translates an operational need — "transport 40,000 lb of cargo 3,000 nmi in under 8 hours" — into quantified Top-Level Aircraft Requirements (TLARs).

**Key activities:**
- Concept of operations (ConOps) development
- Mission profile construction (climb, cruise, reserve, descent segments)
- Requirements decomposition: TLARs → system requirements → subsystem requirements
- Trade-space scoping: turbofan vs turboprop, conventional vs blended-wing-body

**Outputs:** Requirements baseline, ConOps document, preliminary trade study

**Freedom to change:** Maximum — decisions exist only on paper.
```

```{dropdown} Phase 1 — Conceptual design
**Question:** *What should it look like, and can it be sized to meet the requirements?*

The first time a specific aircraft configuration takes shape.
Designers use historical correlations, simple aerodynamic methods, and the sizing/synthesis process (covered in detail below) to establish gross weight, wing loading, thrust loading, and a configuration layout.

**Key activities:**
- Parametric weight estimation (Raymer statistical methods, Torenbeek regression)
- Constraint diagram construction
- Design point selection
- Layout sketching (three-view, inboard profile)
- First-pass performance estimation (Breguet range, climb, T-O field)

**Outputs:** Sized concept, three-view drawing, estimated performance, cost estimate

**Freedom to change:** Very high — changes require only engineering time.
```

```{dropdown} Phase 2 — Preliminary design
**Question:** *Does it actually work? Can the subsystems be integrated?*

The configuration is essentially frozen; it is now refined and validated.
Higher-fidelity aerodynamic, structural, and systems analyses are run for the first time.
Wind tunnel testing begins.

**Key activities:**
- CFD analysis and wind tunnel model testing
- Structural finite-element modelling (FEM)
- Propulsion integration and nacelle design
- Landing gear kinematics, fuel system routing, control surface sizing
- Supplier identification and make/buy decisions

**Outputs:** Preliminary design package, updated weight statement, risk register

**Freedom to change:** Moderate — configuration changes are costly and require re-analysis of all dependent subsystems.
```

```{dropdown} Phase 3 — Detailed design
**Question:** *How exactly is every part made, and how does it assemble?*

Every component is fully defined for manufacturing.
3D CAD models of every part, full stress analysis, fatigue and damage-tolerance substantiation, and qualification test planning are completed.

**Key activities:**
- Full 3D CAD of every part and assembly
- Stress, fatigue, and damage-tolerance analysis
- Manufacturing process planning
- Tooling design
- First-article inspection planning

**Outputs:** Complete production drawings, parts list, assembly instructions

**Freedom to change:** Very low — tooling may exist; any change triggers a formal engineering change order (ECO) and downstream re-qualification.
```

```{dropdown} Phase 4 — Production, test, and in-service support
**Question:** *Does the certified product meet the requirements in the real world?*

First metal is cut.
Flight test articles are built, ground and flight testing verifies the design against requirements, and certification is obtained.
The aircraft enters production and eventually service, with engineering supporting defect resolution and service-life extensions.

**Key activities:**
- Manufacturing and assembly
- Ground vibration testing (GVT), static and fatigue structural tests
- Flight-test programme (envelope expansion, performance, handling qualities, systems)
- Regulatory certification (FAA Part 25 / EASA CS-25 for transport aircraft)
- In-service support, airworthiness directives

**Outputs:** Type Certificate, production aircraft, maintenance manuals
```

```{warning}
The cost of changing a design decision grows exponentially as the design matures.
A configuration change at Phase 1 costs days of engineering time.
The same change at Phase 3 may cost millions of dollars and months of schedule {cite}`raymer2018`.
This asymmetry is why conceptual design, despite its apparent imprecision, is the most strategically consequential phase.
```

## Cost of change across phases


In [ ]:

import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor':      'none',
    'axes.facecolor':        'none',
    'savefig.facecolor':     'none',
    'savefig.transparent':   True,
    'axes.edgecolor':        '#888888',
    'axes.labelcolor':       '#cccccc',
    'text.color':            '#cccccc',
    'xtick.color':           '#cccccc',
    'ytick.color':           '#cccccc',
    'grid.color':            '#555555',
    'font.size':             14,
    'axes.titlesize':        16,
    'axes.labelsize':        14,
    'xtick.labelsize':       13,
    'ytick.labelsize':       13,
    'legend.fontsize':       13,
    'legend.title_fontsize': 13,
    'figure.titlesize':      16,
    'lines.linewidth':       2.0,
    'savefig.bbox':          'tight',
})
OKI = ['#0072B2','#D55E00','#009E73','#E69F00','#56B4E9','#CC79A7','#F0E442','#333333']
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))

# --- Left panel: bar / point chart (log scale) ---
ax = axes[0]
phases = ['Phase 0\nRequirements', 'Phase 1\nConceptual', 'Phase 2\nPreliminary',
          'Phase 3\nDetailed', 'Phase 4\nProduction']
costs = [0.1, 1, 10, 100, 10000]
x = np.arange(len(phases))
BLUE = OKI[0]; RED = OKI[1]

bars = ax.bar(x, costs, color=[BLUE]*len(x), alpha=0.75, zorder=3, width=0.6)
ax.set_yscale('log')
mult_labels = ['×0.1', '×1', '×10', '×100', '×10,000']
for xi, yi, ml in zip(x, costs, mult_labels):
    ax.text(xi, yi * 2.5, ml, fontsize=11, ha='center', va='bottom',
            color=RED, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(phases, fontsize=10)
ax.set_ylabel('Relative cost of a change (log scale)')
ax.set_title('Cost of change by phase')
ax.yaxis.grid(True, which='major', linestyle='--', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# --- Right panel: knowledge vs freedom-to-change ---
ax2 = axes[1]
phase_x = np.linspace(0, 4, 200)
freedom  = 100 * np.exp(-0.7 * phase_x)
knowledge = 100 * (1 - np.exp(-0.9 * phase_x))

ax2.fill_between(phase_x, freedom,   alpha=0.25, color=BLUE,  label='Freedom to change')
ax2.fill_between(phase_x, knowledge, alpha=0.25, color=OKI[2], label='Design knowledge')
ax2.plot(phase_x, freedom,   color=BLUE,  lw=2.5)
ax2.plot(phase_x, knowledge, color=OKI[2], lw=2.5)

phase_labels = ['Ph. 0', 'Ph. 1', 'Ph. 2', 'Ph. 3', 'Ph. 4']
for i, lbl in enumerate(phase_labels):
    ax2.axvline(i, color='#888888', lw=0.7, linestyle=':', alpha=0.6)
    ax2.text(i + 0.05, 3, lbl, fontsize=9, color='#aaaaaa', va='bottom')

ax2.set_xlim(0, 4)
ax2.set_ylim(0, 108)
ax2.set_xlabel('Design phase (0 → 4)')
ax2.set_ylabel('Relative level (%)')
ax2.set_title('Knowledge vs. freedom to change')
ax2.legend(loc='center right', framealpha=0.0)
ax2.set_xticks([])
ax2.spines[['top', 'right']].set_visible(False)

fig.tight_layout(pad=2.0)
plt.show()


*Figure 1.4.1 — (Left) Relative cost of a design change grows by roughly an order of magnitude per phase.
A change that costs one engineering-day at conceptual design may cost ten thousand days — plus re-certification — at production.
(Right) As the design matures, engineering knowledge accumulates while the freedom to change without incurring cost narrows.
The crossing point near Phase 2 represents the "commitment horizon" beyond which major changes become economically prohibitive.*


## Design reviews (gates)

Formal design reviews mark the transition between phases.
They are not rubber stamps: a failed review returns the programme to the previous phase until the issues are resolved.
In defence and space procurement, these gates are contractually binding and often government-audited {cite}`blanchard2011`.

| Review | Full name | Held at | Key question answered |
|--------|-----------|---------|----------------------|
| MDR | Mission Definition Review | Phase 0 exit | Are the requirements complete and consistent? |
| SRR | System Requirements Review | Phase 0/1 boundary | Does the requirements baseline capture the operational need? |
| PDR | Preliminary Design Review | Phase 1/2 boundary | Is the chosen concept feasible and adequately defined? |
| CDR | Critical Design Review | Phase 2/3 boundary | Is the design complete and ready for fabrication? |
| TRR | Test Readiness Review | Phase 3/4 entry | Is the test plan, instrumentation, and safety case approved? |
| FCA/PCA | Functional/Physical Configuration Audit | Phase 4 | Does the built article match the approved design? |

### Technology Readiness Levels (TRL)

A TRL is a nine-point maturity scale developed by NASA and now mandated in US and European defence procurement.
It provides a common language for assessing how close a technology is to fielded use {cite}`mankins2009`.

| TRL | Definition | Typical milestone |
|-----|-----------|-------------------|
| 1 | Basic principles observed | Published research, first journal paper |
| 2 | Technology concept formulated | Analytical study, feasibility paper |
| 3 | Analytical/experimental proof of concept | Lab demonstration, early simulation |
| 4 | Component validated in laboratory | Bench test of key subsystem |
| 5 | Component validated in relevant environment | Wind tunnel, altitude chamber, etc. |
| 6 | System prototype demonstrated in relevant environment | Integrated prototype on test article |
| 7 | System prototype demonstrated in operational environment | First flight of technology demonstrator |
| 8 | System complete and qualified | Flight-test verified, certification pending |
| 9 | Actual system proven in operational environment | In-service, certified product |

```{note}
Programmes that introduce TRL < 6 technologies at CDR (Phase 2/3 boundary) are statistically much more likely to experience cost overruns and schedule delays.
The F-35 programme entered full-scale development with several critical technologies at TRL 4–5, contributing to a $200 billion+ cost growth over its original estimate {cite}`gao2023`.
The lesson is not to avoid new technology, but to mature it in demonstrators *before* integrating it into a programme.
```

## The constraint diagram


The constraint diagram is the central analytical tool of conceptual sizing.
It maps every performance requirement as a boundary curve in the space of two primary design variables:

- **Wing loading** $W/S$ [lb/ft² or N/m²] — gross weight divided by wing area. High $W/S$ means a small wing for the aircraft's weight: good for cruise speed and structural efficiency, bad for low-speed performance and field length.
- **Thrust-to-weight ratio** $T/W$ (or power-to-weight $P/W$) — sea-level static thrust divided by maximum take-off weight. High $T/W$ gives good climb, acceleration, and hot/high performance at the cost of heavier, more fuel-thirsty engines.

Each performance requirement — stall speed, take-off field length, top-of-climb gradient, cruise speed, ceiling — traces a distinct curve in the $T/W$–$W/S$ plane:

$$
\left(\frac{T}{W}\right)_{\text{climb}} = \frac{1}{L/D} + \frac{\sin\gamma}{1} + \frac{qC_{D_0}}{W/S} + k\frac{W/S}{q}
$$

where $q = \tfrac{1}{2}\rho V^2$ is dynamic pressure, $C_{D_0}$ is the parasite drag coefficient, $k = 1/(\pi e AR)$ is the induced drag factor, $e$ is Oswald efficiency, and $\gamma$ is the climb angle.

The take-off constraint ties $T/W$ to $W/S$ through the ground roll equation:

$$
s_{\text{TO}} = \frac{1.21\,(W/S)}{\rho g C_{L_{\max}}\,(T/W - \mu_r)}
$$

The feasible design space is the region where *all* constraints are simultaneously satisfied.
The optimum design point sits at the corner of the feasible region: it meets every requirement with the smallest $T/W$ for a given $W/S$, minimising engine size and fuel burn.

### The rubber engine concept

In early conceptual design, the engine does not yet exist — the designer is free to scale it up or down to match whatever $T/W$ the constraint diagram demands.
This fictitious, infinitely scalable engine is called a **rubber engine**.
It decouples the aircraft sizing from any specific engine model, allowing the designer to optimise the aircraft without being constrained by a discrete catalogue of available powerplants.

Once the design point is fixed, the rubber engine thrust becomes the specification handed to engine manufacturers for a Request for Proposals (RFP), or the designer selects the closest existing engine and adjusts the aircraft slightly to match.


In [ ]:

import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor':      'none',
    'axes.facecolor':        'none',
    'savefig.facecolor':     'none',
    'savefig.transparent':   True,
    'axes.edgecolor':        '#888888',
    'axes.labelcolor':       '#cccccc',
    'text.color':            '#cccccc',
    'xtick.color':           '#cccccc',
    'ytick.color':           '#cccccc',
    'grid.color':            '#555555',
    'font.size':             14,
    'axes.titlesize':        16,
    'axes.labelsize':        14,
    'xtick.labelsize':       13,
    'ytick.labelsize':       13,
    'legend.fontsize':       13,
    'legend.title_fontsize': 13,
    'figure.titlesize':      16,
    'lines.linewidth':       2.0,
    'savefig.bbox':          'tight',
})
OKI = ['#0072B2','#D55E00','#009E73','#E69F00','#56B4E9','#CC79A7','#F0E442','#333333']
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6.0))

WS = np.linspace(20, 140, 400)   # lb/ft^2
BLUE = OKI[0]; RED = OKI[1]; GREEN = OKI[2]; AMBER = OKI[3]; PURPLE = OKI[5]

rho_sl = 0.002377   # slug/ft^3
g      = 32.174      # ft/s^2

# 1. Take-off field length = 1500 ft (typical transport)
# T/W >= 1.21*(W/S) / (rho*g*CL_max*s_TO) + mu_r
CL_max_TO = 2.2
s_TO      = 1500.0
mu_r      = 0.05
TW_TO = 1.21 * WS / (rho_sl * g * CL_max_TO * s_TO) + mu_r
ax.plot(WS, TW_TO, color=BLUE, lw=2.2, label='Take-off (1 500 ft field)')

# 2. Climb gradient (FAR 25 requirement: 2nd-segment climb, OEI, 2.4%)
# T/W >= 1/(L/D) + sin(gamma)  [simplified; assume L/D=8 at climb config]
LD_clb = 8.0
gamma  = np.arcsin(0.024)
TW_clb = np.ones_like(WS) * (1.0 / LD_clb + np.sin(gamma))
ax.plot(WS, TW_clb, color=RED, lw=2.2, label='2nd-segment climb (OEI)')

# 3. Cruise speed at altitude (Vcr=460 kt = 777 ft/s at 35 000 ft)
rho_cr = 0.000738   # slug/ft^3  (35 000 ft, ISA)
V_cr   = 777.0       # ft/s
q_cr   = 0.5 * rho_cr * V_cr**2
CD0    = 0.018
k_ind  = 0.042
TW_cr  = (q_cr * CD0) / (WS) + k_ind * WS / q_cr
ax.plot(WS, TW_cr, color=GREEN, lw=2.2, label='Cruise speed (460 kt, 35 000 ft)')

# 4. Cruise ceiling / service ceiling (simplified — T/W floor)
TW_ceil = np.ones_like(WS) * 0.22
ax.plot(WS, TW_ceil, color=AMBER, lw=2.2, linestyle='--', label='Service ceiling')

# 5. Landing field length (stall constraint → upper bound on W/S)
CL_max_land = 2.8
V_s_max     = 130.0   # kt → ft/s
V_s_fps     = V_s_max * 1.6878
WS_stall    = 0.5 * rho_sl * V_s_fps**2 * CL_max_land
ax.axvline(WS_stall, color=PURPLE, lw=2.2, linestyle='-.', label=f'Stall / landing (W/S ≤ {WS_stall:.0f} lb/ft²)')

# Feasible region (above TO curve, above climb line, above cruise, above ceiling)
TW_lower = np.maximum.reduce([TW_TO, TW_clb, TW_cr, TW_ceil])
WS_feas  = WS[WS <= WS_stall]
TW_feas  = TW_lower[WS <= WS_stall]
ax.fill_between(WS_feas, TW_feas, 0.60, alpha=0.12, color=BLUE, zorder=0)

# Design point (minimum T/W in feasible region)
idx_dp = np.argmin(TW_feas)
WS_dp  = WS_feas[idx_dp]
TW_dp  = TW_feas[idx_dp]
ax.plot(WS_dp, TW_dp, '*', markersize=18, color='#F0E442', markeredgecolor='#888888',
        markeredgewidth=1.0, zorder=5, label=f'Design point ({WS_dp:.0f}, {TW_dp:.3f})')
ax.annotate('Design\npoint', xy=(WS_dp, TW_dp), xytext=(WS_dp + 8, TW_dp + 0.04),
            fontsize=11, color='#F0E442', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#F0E442', lw=1.2))

ax.text(30, 0.195, 'Feasible\nregion', fontsize=12, color='#aaaaaa', alpha=0.9,
        style='italic', ha='left')

ax.set_xlim(20, 140)
ax.set_ylim(0.10, 0.60)
ax.set_xlabel('Wing loading  W/S  [lb/ft²]')
ax.set_ylabel('Thrust-to-weight ratio  T/W')
ax.set_title('Constraint diagram — generic transport aircraft')
ax.legend(loc='upper right', framealpha=0.15, fontsize=11)
ax.grid(True, linestyle='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


*Figure 1.4.2 — Constraint diagram for a generic transport aircraft.
Each curve is derived from a performance requirement; the shaded region is the feasible design space.
The starred design point is where all constraints are simultaneously satisfied with the minimum $T/W$ at the allowable $W/S$.
In practice, the designer would add margins of 5–10% on $T/W$ before finalising the engine specification.*


## The sizing loop and design synthesis

The constraint diagram gives the design point in $T/W$–$W/S$ space, but it does not yet give absolute numbers.
To get the actual gross weight $W_0$, wing area $S$, and required thrust $T$, the designer must solve the sizing equations iteratively — a process called the **sizing loop** or **design spiral**.

### The Breguet range equation as the sizing driver

For a jet aircraft, the range is governed by the Breguet equation:

$$
R = \frac{V}{\text{SFC}} \cdot \frac{L}{D} \cdot \ln\!\left(\frac{W_i}{W_f}\right)
$$

where $W_i / W_f$ is the fuel-fraction ratio (initial to final weight) for the cruise segment.
Rearranging gives the fuel fraction needed to fly a given range at a given $L/D$ and SFC.
Multiplying all segment fuel fractions (taxi, climb, cruise, reserves, descent) gives the total mission fuel fraction $W_f / W_0$.

The gross weight then follows from:

$$
W_0 = \frac{W_{\text{payload}} + W_{\text{crew}}}{1 - W_f/W_0 - W_e/W_0}
$$

where $W_e / W_0$ is the empty-weight fraction, estimated from historical regression as a function of gross weight (a function of $W_0$ itself — hence the iteration).

### The iteration sequence

1. **Guess** $W_0$ from historical analogues.
2. **Compute** $W_e/W_0$ from regression (e.g., Raymer Table 3.1).
3. **Compute** mission fuel fractions for each segment using Breguet (cruise) and empirical values (taxi, climb, descent).
4. **Solve** the gross-weight equation for a new $W_0$.
5. **Repeat** until $W_0$ converges (typically 3–5 iterations).
6. **Use** the converged $W_0$ with the constraint diagram $W/S$ and $T/W$ to find $S$ and $T$.

### Case study: Boeing 777

The Boeing 777 development (launched 1990, first flight 1994) is one of the best-documented applications of systematic conceptual and preliminary design.

| Parameter | Specification | As-designed value |
|-----------|--------------|-------------------|
| Range (ER variant) | 7,725 nmi | 7,725 nmi |
| Payload | 301 passengers (3-class) | 301 |
| Cruise Mach | 0.84 | 0.84 |
| MTOW | — | 656,000 lb (297,600 kg) |
| Wing area | — | 4,605 ft² (427.8 m²) |
| Wing loading | — | 142 lb/ft² (694 N/m²) |
| T/W (at MTOW) | — | 0.31 (GE90-115B) |
| Aspect ratio | — | 8.68 |
| Empty weight fraction | — | 0.527 |

The 777 design team used a constraint diagram to balance the conflicting demands of long-range efficiency (wanting high $W/S$ to reduce wetted area and induced drag at cruise) against ETOPS diversion field requirements and hot/high performance at airports like Denver (wanting moderate $W/S$).
The selected $W/S pprox 142$ lb/ft² represented the corner of the feasible region at that $T/W$.

Boeing's "Working Together" programme involved direct airline input throughout preliminary design, resulting in more than 1,000 design changes based on customer feedback before CDR — a model of concurrent engineering that compressed the development schedule relative to earlier programmes {cite}`norris1996`.

```{tip}
For your own sizing exercises, start with a guess of $W_0$ from a comparable aircraft (same class, similar range), use Raymer's empty-weight regression for $W_e/W_0$, and run the loop three times.
If the loop diverges, your mission requirements may be incompatible — check whether the fuel fraction exceeds 0.50 (a sign the design is fuel-dominated and a different propulsion concept is needed).
```


```{bibliography}
:filter: docname in docnames
```
